# Demo di Spark Structured Streaming

## Introduzione
Questo notebook mostra una applicazione minimale di **Spark Structured Streaming**.
Structured Streaming è un motore di stream processing scalabile e fault-tolerant, costruito sopra il motore Spark SQL: una computazione in streaming si esprime nello stesso modo di una computazione batch su dati statici.

### Obiettivi
1.  Inizializzare una `SparkSession` collegata a uno Spark Master.
2.  Leggere uno stream di dati da una sorgente generata (`rate`).
3.  Applicare una trasformazione di base ai dati.
    - Confrontare la **stessa trasformazione** su un DataFrame statico (batch) e su uno in streaming, e vedere quali operazioni batch non sono supportate sugli stream.
4.  Scrivere lo stream elaborato in una tabella in memoria per ispezionarlo.
5.  Interrogare periodicamente i risultati.
6.  Eseguire aggregazioni con **finestre Tumbling**.
7.  Eseguire aggregazioni con **finestre Sliding**.
8.  Gestire dati fuori ordine con **event time e watermark**.
9.  Confrontare le **modalità di output** (`append`, `update`, `complete`).
10. Controllare quando parte un micro-batch con i **trigger**.

## Step 1: Inizializzare la SparkSession

La `SparkSession` è il punto di ingresso per programmare Spark con le API Dataset e DataFrame.
Qui la configuriamo per collegarsi allo Spark Master definito nel `docker-compose.yml` (servizio `spark-master` sulla porta `7077`).

In [ ]:
import os
import time
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, window, avg

print(f"Versione PySpark: {pyspark.__version__}")

# Inizializza la SparkSession. Il driver gira in questo container: gli executor sul worker
# lo raggiungono tramite il suo hostname ("jupyter", definito nel docker-compose.yml).
spark = SparkSession.builder \
    .appName("StructuredStreamingDemo") \
    .master("spark://spark-master:7077") \
    .config("spark.driver.host", "jupyter") \
    .config("spark.sql.shuffle.partitions", "2") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print(f"SparkSession inizializzata. Versione Spark: {spark.version}")

## Step 2: Creare un DataFrame in streaming

Usiamo la sorgente `rate`, una sorgente di test fornita da Spark che genera dati al ritmo indicato (righe al secondo). Ogni riga generata contiene un `timestamp` e un `value` (un long auto-incrementale).

In [ ]:
# Crea un DataFrame in streaming con la sorgente rate
# rowsPerSecond: quante righe generare al secondo
rate_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 1) \
    .load()

print("DataFrame in streaming creato. Schema:")
rate_df.printSchema()

## Step 3: Applicare le trasformazioni

Structured Streaming permette di applicare le normali operazioni sui DataFrame. Qui aggiungiamo semplicemente una colonna `value_plus_10` che somma 10 al `value` generato.

In [ ]:
# Trasformazione semplice: somma 10 al valore
transformed_df = rate_df.withColumn("value_plus_10", col("value") + 10)

print("Trasformazione definita.")

## Step 3b: Batch vs Streaming — Una API, due modalità di esecuzione

Structured Streaming usa la **stessa API DataFrame** per batch e streaming.
Una query batch diventa una query in streaming passando da `read()`/`write()` a `readStream()`/`writeStream()`:
la *logica* resta la stessa, cambia solo l'*esecuzione* (una tantum contro incrementale e continua).

Per mostrarlo, scriviamo la trasformazione **una sola volta**, come semplice funzione Python, e la applichiamo a:
- un DataFrame **statico** (una tabella finita già esistente), e
- il DataFrame in **streaming** `rate_df` (una tabella illimitata che continua a crescere).

Il DataFrame statico ha lo stesso schema della sorgente `rate` (`timestamp`, `value`), quindi lo stesso codice funziona su entrambi.

In [ ]:
from pyspark.sql.functions import current_timestamp

# La logica di business, scritta UNA volta e indipendente da batch/streaming
def add_ten_and_keep_even(df):
    return (df
            .withColumn("value_plus_10", col("value") + 10)
            .filter(col("value") % 2 == 0))

# Un DataFrame statico con lo stesso schema della sorgente 'rate'
static_df = spark.range(0, 10).select(current_timestamp().alias("timestamp"), col("id").alias("value"))

static_result = add_ten_and_keep_even(static_df)      # batch
streaming_result = add_ten_and_keep_even(rate_df)     # streaming

print("static_result.isStreaming    =", static_result.isStreaming)
print("streaming_result.isStreaming =", streaming_result.isStreaming)

I due DataFrame hanno lo **stesso schema** e lo **stesso piano logico** (filtro + proiezione).
La differenza emerge solo quando chiediamo il risultato:
- il DataFrame batch viene calcolato una volta, appena chiamiamo un'azione come `show()`;
- il DataFrame in streaming descrive una computazione che non finisce mai, quindi può solo essere *avviato* con `writeStream.start()`.

In [ ]:
# Stesso schema
static_result.printSchema()
streaming_result.printSchema()

# Batch: un'azione calcola subito l'intero risultato (finito)
static_result.show()

### Cosa succede se trattiamo uno stream come una tabella?

Chiamare un'azione batch come `show()` o `count()` su un DataFrame in streaming fallisce:
una tabella illimitata non ha un risultato "finale" da restituire.

In [ ]:
try:
    streaming_result.show()
except Exception as e:
    print(f"{type(e).__name__}:")
    print(str(e).splitlines()[0])

### Non tutte le operazioni batch sono supportate in streaming

Alcune operazioni banali in batch non si possono calcolare in modo incrementale su uno stream illimitato.
Un esempio tipico è l'**ordinamento senza aggregazione**: per ordinare uno stream Spark dovrebbe aspettare *tutti* i dati, che non arrivano mai.

La query qui sotto è perfettamente valida sul DataFrame statico, ma Spark rifiuta di avviarla sullo stream.
Il controllo avviene quando la query viene **avviata**, non quando si definisce la trasformazione.

In [ ]:
# Batch: l'ordinamento funziona
add_ten_and_keep_even(static_df).orderBy(col("value").desc()).show(3)

# Streaming: lo stesso ordinamento viene rifiutato all'avvio della query
sorted_stream = streaming_result.orderBy(col("value").desc())
try:
    bad_query = sorted_stream.writeStream \
        .outputMode("append") \
        .format("memory") \
        .queryName("sorted_table") \
        .start()
    bad_query.stop()
except Exception as e:
    print(f"{type(e).__name__}:")
    print(str(e).splitlines()[0])

**Punti chiave**
- Stessa API e stessa logica: si può sviluppare e testare una trasformazione su dati statici, poi eseguirla su uno stream.
- Esecuzione diversa: una query in streaming gira in modo incrementale e non termina mai, quindi le azioni batch (`show()`, `count()`, `collect()`) non sono permesse.
- Alcune operazioni (ad es. l'ordinamento senza aggregazione, come vedremo più avanti con le finestre in modalità *Complete*) sono permesse solo in casi specifici.

## Step 4: Avviare la query in streaming

Definiamo il sink di output e avviamo la query.
- **Output Mode `append`**: nel sink vengono scritte solo le nuove righe aggiunte alla Result Table dall'ultimo trigger.
- **Formato `memory`**: l'output è salvato in una tabella in memoria, interrogabile con SQL. **Nota**: serve solo per il debug e non va usato in produzione, perché può causare OOM.
- **Query Name**: il nome della tabella in memoria (`rate_table`).

In [ ]:
# Avvia la query in streaming, scrivendo nel sink memory
query = transformed_df.writeStream \
    .outputMode("append") \
    .format("memory") \
    .queryName("rate_table") \
    .start()

print("Query in streaming avviata...")

## Step 5: Monitorare i risultati

Poiché i dati vengono scritti nella tabella `rate_table` in memoria, possiamo interrogarla con il normale Spark SQL. Eseguiamo un ciclo che interroga la tabella ogni 2 secondi per vedere arrivare i dati.

In [ ]:
# Interroga periodicamente la tabella in memoria per mostrare i risultati
print("Interrogo la tabella in memoria per 10 secondi...")
try:
    for i in range(5):
        print(f"\n--- Interrogazione {i+1} ---")
        spark.sql("SELECT * FROM rate_table ORDER BY timestamp DESC LIMIT 5").show()
        time.sleep(2)
except KeyboardInterrupt:
    print("Monitoraggio interrotto dall'utente.")

## Step 6: Fermare la query

Ricordati sempre di fermare la query in streaming quando hai finito, per liberare le risorse.

In [ ]:
# Ferma la query
if 'query' in locals() and query.isActive:
    query.stop()
    print("Query fermata.")
else:
    print("Nessuna query attiva da fermare.")

## Step 7: Aggregazione con finestre Tumbling

Le **finestre tumbling** sono intervalli di tempo di dimensione fissa, non sovrapposti e contigui.
Qui calcoliamo il valore medio per ogni finestra di **10 secondi**.

In [ ]:
# Finestra Tumbling: 10 secondi
tumbling_window_df = rate_df \
    .groupBy(window(col("timestamp"), "10 seconds")) \
    .agg(avg("value").alias("avg_value")) \
    .orderBy("window")

print("Aggregazione con finestra tumbling definita.")
tumbling_window_df.printSchema()

In [ ]:
# Avvia la query con finestra tumbling
# NB: usiamo "complete" perché la query ha un orderBy sull'aggregazione (permesso solo in Complete)
# e perché non abbiamo un watermark (senza watermark "append" non è ammesso: lo vedremo allo Step 9).
tumbling_query = tumbling_window_df.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("tumbling_table") \
    .start()

print("Query con finestra tumbling avviata...")

In [ ]:
print("Monitoraggio delle finestre tumbling per 15 secondi...")
try:
    for i in range(5):
        print(f"\n--- Interrogazione tumbling {i+1} ---")
        spark.sql("SELECT * FROM tumbling_table ORDER BY window DESC").show(truncate=False)
        time.sleep(3)
except KeyboardInterrupt:
    print("Monitoraggio interrotto.")

# Ferma la query tumbling
if 'tumbling_query' in locals() and tumbling_query.isActive:
    tumbling_query.stop()
    print("Query tumbling fermata.")

## Step 8: Aggregazione con finestre Sliding

Le **finestre sliding** hanno dimensione fissa e possono sovrapporsi. Si definiscono con una durata della finestra e una durata dello scorrimento (slide).
Qui calcoliamo il valore medio su una **finestra di 10 secondi**, che scorre ogni **5 secondi**.

In [ ]:
# Finestra Sliding: durata 10 secondi, slide di 5 secondi
sliding_window_df = rate_df \
    .groupBy(window(col("timestamp"), "10 seconds", "5 seconds")) \
    .agg(avg("value").alias("avg_value")) \
    .orderBy("window")

print("Aggregazione con finestra sliding definita.")
sliding_window_df.printSchema()

In [ ]:
# Avvia la query con finestra sliding
sliding_query = sliding_window_df.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("sliding_table") \
    .start()

print("Query con finestra sliding avviata...")

In [ ]:
print("Monitoraggio delle finestre sliding per 15 secondi...")
try:
    for i in range(5):
        print(f"\n--- Interrogazione sliding {i+1} ---")
        spark.sql("SELECT * FROM sliding_table ORDER BY window DESC").show(truncate=False)
        time.sleep(3)
except KeyboardInterrupt:
    print("Monitoraggio interrotto.")

# Ferma la query sliding
if 'sliding_query' in locals() and sliding_query.isActive:
    sliding_query.stop()
    print("Query sliding fermata.")

## Step 9: Event time, dati fuori ordine e Watermark

Finora le finestre usavano il `timestamp` della sorgente `rate`, che viene assegnato **nel momento in cui la riga viene generata**: event time e processing time coincidono e i dati arrivano sempre in ordine.
Nella realtà l'**event time** (quando l'evento è accaduto) e il **processing time** (quando il motore lo vede) differiscono, e i dati possono arrivare **fuori ordine**.

Il **watermark** è la dichiarazione "non mi aspetto dati con event time più vecchio di *T*", con
`T = massimo event time visto - soglia`. Serve a:
- decidere **quando una finestra è definitiva** e può essere emessa;
- **scartare lo stato** delle finestre chiuse (la memoria non cresce all'infinito);
- scartare (o ignorare) i dati che arrivano troppo in ritardo.

Per simularlo creiamo un event time "sporco": sottraiamo al `timestamp` un ritardo casuale da 0 a 29 secondi, così le righe arrivano con event time fuori ordine.

In [ ]:
from pyspark.sql import functions as F

# Stream con event time fuori ordine: ogni riga "è accaduta" da 0 a 29 secondi prima di quando la vediamo
events_df = rate_df.select(
    col("value"),
    col("timestamp").alias("processing_time"),
    (col("timestamp").cast("double") - F.floor(F.rand() * 30)).cast("timestamp").alias("event_time")
)

events_df.printSchema()

### 9a. Senza watermark, `append` con aggregazione non è ammesso

In modalità `append` ogni riga viene emessa **una sola volta e per sempre**. Una finestra ancora aperta potrebbe ricevere altri dati e cambiare, quindi Spark la può emettere solo quando sa che è finale: serve un watermark.
Senza watermark la query viene rifiutata all'avvio. (Per questo agli step 7 e 8 abbiamo usato `complete`.)

In [ ]:
no_watermark_df = events_df \
    .groupBy(window(col("event_time"), "10 seconds")) \
    .agg(F.count("*").alias("n_eventi"))

try:
    no_watermark_df.writeStream \
        .outputMode("append") \
        .format("memory") \
        .queryName("no_watermark_table") \
        .start()
except Exception as e:
    print(f"{type(e).__name__}:")
    print(str(e).splitlines()[0])

### 9b. Con watermark, `append` emette le finestre solo quando sono definitive

Dichiariamo `withWatermark("event_time", "10 seconds")`: Spark tollera fino a 10 secondi di ritardo rispetto al massimo event time visto.
- Una finestra `[t, t+10)` viene **emessa in `append`** solo quando il watermark supera `t+10`: per questo i risultati compaiono con ritardo (finestra + soglia).
- Le righe che cadono in una finestra **già chiusa dal watermark** sono considerate in ritardo e vengono scartate (`numRowsDroppedByWatermark`, può restare a 0 per qualche micro-batch).
- Lo **stato** delle finestre chiuse viene eliminato (`numRowsTotal` non cresce all'infinito, a differenza della modalità `complete`).

In [ ]:
# NB: il watermark va dichiarato sulla stessa colonna usata nella finestra
watermarked_counts = events_df \
    .withWatermark("event_time", "10 seconds") \
    .groupBy(window(col("event_time"), "10 seconds")) \
    .agg(F.count("*").alias("n_eventi"))

watermark_query = watermarked_counts.writeStream \
    .outputMode("append") \
    .format("memory") \
    .queryName("watermark_table") \
    .start()

print("Query con watermark (append) avviata...")

In [ ]:
print("Monitoraggio per ~40 secondi: le finestre compaiono solo dopo che il watermark le supera.")
try:
    for i in range(8):
        print(f"\n--- Interrogazione watermark {i+1} ---")
        p = watermark_query.lastProgress
        if p:
            state = p["stateOperators"][0] if p.get("stateOperators") else {}
            print("Watermark corrente      :", p.get("eventTime", {}).get("watermark"))
            print("Righe di stato (aperte) :", state.get("numRowsTotal"))
            print("Righe scartate (tardive):", state.get("numRowsDroppedByWatermark"))
        spark.sql("SELECT * FROM watermark_table ORDER BY window").show(truncate=False)
        time.sleep(5)
except KeyboardInterrupt:
    print("Monitoraggio interrotto.")

if watermark_query.isActive:
    watermark_query.stop()
    print("Query watermark fermata.")

## Step 10: Modalità di output a confronto (`append`, `update`, `complete`)

La modalità di output definisce **cosa viene scritto nel sink a ogni trigger**. Sulla stessa aggregazione con watermark:

| Modalità | Cosa scrive a ogni trigger | Vincoli |
|---|---|---|
| `append` (default) | solo le righe **nuove e definitive** della Result Table, emesse una sola volta | con aggregazioni serve il watermark |
| `update` | solo le righe **cambiate** dall'ultimo trigger (una finestra può comparire più volte, con valori aggiornati) | non ammette `orderBy`; con watermark lo stato viene ripulito |
| `complete` | **l'intera** Result Table a ogni trigger | solo per aggregazioni; lo stato non si può mai scartare |

Per vedere cosa riceve il sink a ogni trigger usiamo `foreachBatch`, che consegna a una funzione il DataFrame del micro-batch (e il suo `batch_id`).
Raccogliamo i risultati in una lista e li stampiamo alla fine.

In [ ]:
update_log = []   # (batch_id, [righe cambiate in questo micro-batch])

def log_update_batch(batch_df, batch_id):
    changed = [f"{r['window'].start:%H:%M:%S}-{r['window'].end:%H:%M:%S} -> n_eventi={r['n_eventi']}"
               for r in batch_df.orderBy("window").collect()]
    update_log.append((batch_id, changed))

update_query = watermarked_counts.writeStream \
    .outputMode("update") \
    .foreachBatch(log_update_batch) \
    .trigger(processingTime="5 seconds") \
    .start()

print("Query in modalità update avviata per ~30 secondi...")
time.sleep(30)
update_query.stop()

for batch_id, changed in update_log:
    print(f"\nMicro-batch {batch_id}: {len(changed)} righe cambiate")
    for line in changed:
        print("   ", line)

**Cosa osservare**
- In `update` una stessa finestra compare in **più micro-batch** con `n_eventi` crescente, finché il watermark non la chiude: il sink vede ogni aggiornamento.
- In `append` (Step 9b) la stessa finestra comparirebbe **una sola volta**, già col valore finale e in ritardo.
- In `complete` ogni micro-batch conterrebbe **tutte** le finestre dall'inizio dello stream.

## Step 11: Trigger — quando parte il micro-batch

Il **trigger** definisce quando inizia il prossimo micro-batch:

| Trigger | Sintassi | Comportamento |
|---|---|---|
| Default | *(nessuno)* | un nuovo micro-batch parte appena finisce il precedente |
| Intervallo fisso | `.trigger(processingTime="5 seconds")` | un micro-batch ogni intervallo; se non ci sono dati nuovi non viene eseguito nulla |
| Available-now | `.trigger(availableNow=True)` | elabora tutti i dati disponibili (anche in più micro-batch) e poi si ferma; utile per job batch incrementali, sostituisce il deprecato `once` |
| Continuous | `.trigger(continuous="1 second")` | sperimentale/legacy, garanzia at-least-once |
| Real-time | *(Spark 4.1+)* | latenza di millisecondi con la stessa API |

Qui confrontiamo **default** e **intervallo fisso** sullo stesso stream stateless: la funzione `run_logged` registra, per ogni micro-batch, l'ora, il numero di righe e l'intervallo di `value` elaborato.

In [ ]:
def run_logged(seconds, **trigger):
    """Esegue rate_df per `seconds` secondi e registra ogni micro-batch (ora, n righe, min/max value)."""
    log = []

    def on_batch(batch_df, batch_id):
        r = batch_df.agg(F.count("*").alias("n"), F.min("value").alias("lo"), F.max("value").alias("hi")).first()
        log.append((batch_id, time.strftime("%H:%M:%S"), r["n"], r["lo"], r["hi"]))

    writer = rate_df.writeStream.foreachBatch(on_batch)
    if trigger:
        writer = writer.trigger(**trigger)                       # es. processingTime="5 seconds"

    q = writer.start()
    time.sleep(seconds)
    q.stop()
    return log

def print_log(title, log):
    print(f"\n{title}")
    print(f"{'batch':>6} {'ora':>9} {'righe':>6} {'value min':>10} {'value max':>10}")
    for b, t, n, lo, hi in log:
        print(f"{b:>6} {t:>9} {n:>6} {str(lo):>10} {str(hi):>10}")

# 1) Trigger di default: micro-batch ravvicinati, con poche righe ciascuno
print_log("Trigger di default (8 s)", run_logged(8))

# 2) Intervallo fisso: un micro-batch ogni 5 secondi, con più righe ciascuno
print_log("processingTime='5 seconds' (16 s)", run_logged(16, processingTime="5 seconds"))

**Cosa osservare**: con il trigger di default i micro-batch sono molti e piccoli (latenza minima, più overhead); con `processingTime="5 seconds"` sono uno ogni ~5 secondi e contengono ~5 righe ciascuno (meno overhead, latenza maggiore).

## Pulizia

Se una cella è stata interrotta a metà, qualche query può essere rimasta attiva e occupare i core del worker, bloccando le query successive.
Questa cella ferma tutte le query in streaming ancora attive.

In [ ]:
for q in spark.streams.active:
    print("Fermo la query:", q.name or q.id)
    q.stop()

## Riepilogo

| Concetto | Dove |
|---|---|
| API unificata batch/streaming, limiti sugli stream | Step 3b |
| Micro-batch, sink, Result Table | Step 4-5 |
| Finestre tumbling e sliding | Step 7-8 |
| Event time, dati fuori ordine, watermark | Step 9 |
| Modalità di output `append` / `update` / `complete` | Step 7-10 |
| Trigger | Step 11 |